In [ ]:
# Building a basic selector ML algorithm for picking simulation strategy with inferQ data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
from utils import * 

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

base = Path("clean_tables")
PRIMARY_KEY = "RowKey"

# 1. Load feature tables
dynamic_df = pd.read_csv(base / "dynamic_metrics.csv")
dynamic_df = clean_dataframe(dynamic_df, "statevector_saved_von_neumann_entropy")

static_df  = pd.read_csv(base / "static_metrics.csv")
graph_df   = pd.read_csv(base / "graph_metrics.csv")
# sql_df    = pd.read_csv(base / "sql_metrics.csv")
qiskit_df  = pd.read_csv(base / "qiskit_simulation.csv")

# 2. Merge feature tables first (numeric features)
feature_df = dynamic_df.merge(static_df, on=PRIMARY_KEY)\
                       .merge(graph_df, on=PRIMARY_KEY)\
                        #   .merge(sql_df, on=PRIMARY_KEY)

# 3. FEATURES = numeric columns only (excluding PRIMARY_KEY)
FEATURES = feature_df.select_dtypes(include=[np.number]).columns.tolist()

# 4. Keep only memory_usage / execution_time columns from Qiskit
keep_suffixes = ["memory_usage", "execution_time"]
qiskit_cols = [PRIMARY_KEY] + [c for c in qiskit_df.columns if any(c.endswith(s) for s in keep_suffixes)]
qiskit_df = qiskit_df[qiskit_cols]

# 5. Merge feature_df with Qiskit simulation columns to get final ML-ready df
df = feature_df.merge(qiskit_df, on=PRIMARY_KEY)

df_clean = pd.read_parquet("training_data/selector_training_data.parquet")

# 6. Keep rows where at least one automatic_* column is not NaN
auto_cols = [c for c in df.columns if c.startswith("automatic_")]
df = df[df[auto_cols].notna().any(axis=1)].copy()

# 7. Add target: at least one simulation worked
sim_cols = [c for c in qiskit_df.columns if c != PRIMARY_KEY]
df["at_least_one_sim"] = df[sim_cols].notna().any(axis=1)

print("Rows after filtering automatic simulation:", len(df))
print("Number of numeric feature columns:", len(FEATURES))
print("FEATURES sample:", FEATURES)


In [ ]:
METHODS = [
    # "statevector_saved",
    "statevector",
    # "stabilizer",
    "density_matrix",
    "matrix_product_state",
    "extended_stabilizer",
    # "unitary",
    # "automatic"
]
# def best_method(row):
#     costs = {}
#     for m in METHODS:
#         t = row.get(f"{m}_execution_time")
#         mem = row.get(f"{m}_memory_usage")
#         if pd.isna(t) or pd.isna(mem):
#             continue
#         costs[m] = t
#     return min(costs, key=costs.get) if costs else np.nan
# df_dropped = df.copy(deep=True)
# df_dropped["best_method"] = df_dropped.apply(best_method, axis=1)
# df_clean = df_dropped.dropna(subset=["best_method"])
print(len(df_clean))

# Feature Analysis

In [ ]:
# from utils import PlotHistPercentBest

# PlotHistPercentBest(METHODS, FEATURES, df_clean)

In [ ]:
# from utils import PlotHeatMapAvg
# PlotHeatMapAvg(METHODS, FEATURES, df_clean)

# Best Method

In [ ]:
import matplotlib.pyplot as plt

percentages = 100 * df_clean["best_method"].value_counts() / len(df_clean)

ax = percentages.plot(kind="bar")
plt.xlabel("Best simulation method")
plt.ylabel("Percentage of circuits")
plt.title("Distribution of Best Simulation Method (%)")
plt.ylim(0, 100)

# Add percentage labels on top of bars
for i, pct in enumerate(percentages):
    ax.text(i, pct + 1, f"{pct:.1f}%", ha='center')

plt.tight_layout()
plt.show()



In [ ]:
total = len(df_clean)
print(df_clean["best_method"].value_counts())
print(len(df_clean))
print(df_clean["best_method"].value_counts()/total)

In [ ]:
import pandas as pd
import numpy as np

rank_counts = pd.DataFrame(0, index=METHODS, columns=range(1, len(METHODS)+1))

for _, row in df_clean.iterrows():
    # Build a dict of method: cost for this row
    costs = {}
    for m in METHODS:
        t = row.get(f"{m}_execution_time")
        mem = row.get(f"{m}_memory_usage")
        if pd.isna(t) or pd.isna(mem):
            continue
        costs[m] = t

    if not costs:
        continue  # skip rows with no valid methods

    # Rank methods by cost (ascending = smaller is better)
    ranked_methods = sorted(costs, key=costs.get)  # best -> worst

    # Only tally ranks for methods that exist
    for m in ranked_methods:
        actual_rank = ranked_methods.index(m) + 1
        rank_counts.loc[m, actual_rank] += 1

print(rank_counts)


In [ ]:
X = df_clean[FEATURES].select_dtypes(include=[np.number])
y = df_clean["best_method"]

# PCA 

In [ ]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from sklearn.preprocessing import StandardScaler
# from sklearn.decomposition import PCA

# # Standardize features
# X_PCA = StandardScaler().fit_transform(X)

# print(len(FEATURES))

# # Fit PCA to inspect explained variance
# pca = PCA()
# pca.fit(X_PCA)

# # Cumulative explained variance
# cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

# # Plot cumulative explained variance
# plt.figure()
# plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o')
# plt.xlabel("Number of Principal Components")
# plt.ylabel("Cumulative Explained Variance")
# plt.grid(True)
# plt.show()

# # Number of components needed for 95% variance
# n_components = np.argmax(cumulative_variance >= 0.89) + 1
# print(f"Number of components needed for 95% variance: {n_components}")

# # Apply PCA with 15 components and overwrite X
# pca_n = PCA(n_components)
# X = pd.DataFrame(
#     pca_n.fit_transform(X_PCA),
#     columns=[f"PC{i+1}" for i in range(n_components)],
#     index=X.index
# )

# print(X.shape)


# Training Models

In [ ]:
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

TARGET = "best_method"
RANDOM_STATE = 42

X = df_clean[FEATURES]
y = df_clean[TARGET]

y_original = df_clean[TARGET]


X_train, X_test, y_train, y_test, y_orig_train, y_orig_test = train_test_split(
    X, y, y_original,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)
y.value_counts(normalize=True)


In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
def PlotConfusionReport(y_test, y_pred, method):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        labels=METHODS,
        display_labels=METHODS,
        ax=axes[0],
        xticks_rotation=45,
        normalize=None
    )
    axes[0].set_title("Confusion Matrix (Counts)")
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        labels=METHODS,
        display_labels=METHODS,
        ax=axes[1],
        xticks_rotation=45,
        normalize="true"
    )
    axes[1].set_title("Confusion Matrix (Normalized)")
    plt.tight_layout()
    plt.show()

    # Compute normalized confusion matrix
    cm_norm = confusion_matrix(y_test, y_pred, labels=METHODS, normalize='true')
    
    # Count how many classes have diagonal > 0.7
    correct_diag = np.diag(cm_norm)
    above_70 = np.sum(correct_diag > 0.7)
    
    print(f"Number of classes with >70% correct predictions: {above_70} out of {len(METHODS)}")
    for idx, val in enumerate(correct_diag):
        print(f"{METHODS[idx]}: {val:.2f}")

# Naive Bayes Gaussian

In [ ]:
from sklearn.naive_bayes import GaussianNB

modelNaiveBayes = GaussianNB()
modelNaiveBayes.fit(X_train, y_train)
y_predNaiveBayes = modelNaiveBayes.predict(X_test)

print(classification_report(y_test, y_predNaiveBayes))
PlotConfusionReport(y_test, y_predNaiveBayes, "Naive Bayes")

# Logisitic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

modelLogisticRegression = LogisticRegression(
    random_state=42,
    class_weight="balanced",
)

modelLogisticRegression.fit(X_train, y_train)
y_predLogisticRegression = modelLogisticRegression.predict(X_test)

print(classification_report(y_test, y_predLogisticRegression))

PlotConfusionReport(y_test, y_predLogisticRegression, "Logistic Regression")


# KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

modelKNN = KNeighborsClassifier(
    n_neighbors=5
)

modelKNN.fit(X_train, y_train)
y_predKNN = modelKNN.predict(X_test)

print(classification_report(y_test, y_predKNN))
PlotConfusionReport(y_test, y_predKNN, "K-Nearest Neighbors")

# Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

modelDecisionTree = DecisionTreeClassifier(
    max_depth=5,
    random_state=42,
    class_weight="balanced",
)

modelDecisionTree.fit(X_train, y_train)
y_predDecisionTree = modelDecisionTree.predict(X_test)

print(classification_report(y_test, y_predDecisionTree))
PlotConfusionReport(y_test, y_predDecisionTree, "Decision Tree")

# Random Forest 

In [ ]:
from sklearn.ensemble import RandomForestClassifier
modelRandomForest = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=10,
    n_jobs=-1,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

modelRandomForest.fit(X_train, y_train)
y_predRandomForest = modelRandomForest.predict(X_test)

print(classification_report(
    y_test, y_predRandomForest,
    labels=METHODS,
    target_names = METHODS
))

PlotConfusionReport(y_test, y_predRandomForest, "Random Forest")

# XG Boost

In [ ]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import numpy as np

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

num_classes = len(le.classes_)

# model = XGBClassifier(
#     n_estimators=400,
#     max_depth=12,
#     learning_rate=0.05,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     min_child_weight=10,
#     objective="multi:softmax",   # softmax since we want class labels
#     num_class=num_classes,
#     n_jobs=-1,
#     random_state=RANDOM_STATE,
#     eval_metric="mlogloss"
# )

# modelXGBoost = XGBClassifier(
#     n_estimators=100,            # fewer trees → faster, less overfitting
#     max_depth=4,                 # shallower → better generalization
#     learning_rate=0.1,           # slightly higher → faster learning
#     subsample=0.8,               # keep as is
#     colsample_bytree=0.8,        # keep as is
#     min_child_weight=1,          # smaller → allows splits for small classes
#     objective="multi:softmax",
#     num_class=num_classes,
#     n_jobs=-1,
#     random_state=RANDOM_STATE,
#     eval_metric="mlogloss",
#     scale_pos_weight=None         # can handle class imbalance manually if needed
# )

# ==================== Balanced XGBoost ====================

# Compute class weights based on frequency
class_counts = df_clean["best_method"].value_counts()
total = len(df_clean)
class_weights = {cls: total / count for cls, count in class_counts.items()}

# Map sample weights to training set (using numeric encoded labels)
weights = np.array([class_weights[le.inverse_transform([y])[0]] for y in y_train_enc])

modelXGBoost = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=1,
    objective="multi:softmax",
    num_class=num_classes,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    eval_metric="mlogloss"
)

# Fit using sample weights
modelXGBoost.fit(X_train, y_train_enc, sample_weight=weights)

# Predict numeric labels
y_pred_encXG = modelXGBoost.predict(X_test)

# Convert back to original class names
y_predXGBoost = le.inverse_transform(y_pred_encXG)

print(classification_report(y_test, y_predXGBoost))
PlotConfusionReport(y_test, y_predXGBoost, "XGBoost Classifier")


# SVC 

In [ ]:
# print(X_train.isna().sum().sort_values(ascending=False))

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

modelSVC = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(
        C=10.0,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        max_iter=5000
    ))
])

modelSVC.fit(X_train, y_train_enc)

y_pred_encSVC = modelSVC.predict(X_test)
y_predSVC = le.inverse_transform(y_pred_encSVC)

print(classification_report(
    y_test,
    y_predSVC,
    labels=le.classes_,
    target_names=le.classes_
))


PlotConfusionReport(y_test, y_predSVC, "Linear SVM")


In [ ]:
# Let us make subplots for all the models in one figure instead of seperate

model_to_y_pred = {
    "Random Forest": y_predRandomForest,
    "XGBoost Classifier": y_predXGBoost,
    # "Naive Bayes": y_predNaiveBayes,
    "Logistic Regression": y_predLogisticRegression,
    "K-Nearest Neighbors": y_predKNN,
    "Decision Tree": y_predDecisionTree,
    "Linear SVM": y_predSVC
}


import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
import math

def PlotConfusionReports(y_test, predictions_dict, methods):
    n_models = len(predictions_dict)
    n_rows = math.ceil(np.sqrt(n_models))
    n_cols = math.ceil(n_models / n_rows)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = np.array(axes).reshape(-1)

    for ax, (model_name, y_pred) in zip(axes, predictions_dict.items()):
        acc = accuracy_score(y_test, y_pred)

        ConfusionMatrixDisplay.from_predictions(
            y_test,
            y_pred,
            labels=methods,
            display_labels=methods,
            normalize="true",
            ax=ax,
            xticks_rotation=45
        )

        ax.set_title(f"{model_name}\nAccuracy: {acc:.3f}")
        

    for ax in axes[len(predictions_dict):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


PlotConfusionReports(y_test, model_to_y_pred, METHODS)


In [ ]:
# Now let us compare against the accuracy of automatic
METHODS.append("automatic")

df_auto = df_dropped
print(len(df_auto))

df_auto["best_method"] = df_auto.apply(best_method, axis=1)
df_auto = df_auto.dropna(subset=["best_method"])

total = len(df_auto)

counts = df_auto["best_method"].value_counts()
percentages = counts / total * 100
summary_df = pd.DataFrame({
    "count": counts,
    "percentage": percentages
})
print(summary_df)
METHODS.remove("automatic")